# Gold dimension: `dim_project_status`

## Purpose

The DPWH project statuses that `fact_project_snapshot` joins to, with the approved standardized status when one exists.

## Grain

One source status per source system and status mapping version, plus key `0`. Uniqueness: `source_system`, `source_status`, `status_mapping_version`.

## Keys

Primary key `status_key`. Recipe: `HASH` of `PROJECT_STATUS`, the source system, the source status, and the mapping version (D-33).

## Key 0 member

"Missing or conflicting status". Facts use it when the source status is missing.

## Inputs

- `02-silver.silver_project` (distinct `status_raw`)
- `02-silver.config_project_status_mapping` (active, approved rows only)
- `04-validation.silver_dq_results` and `silver_config_dq_results`
- `01-bronze.load_log` (load time only)

## Outputs

`03-gold.dim_project_status`, loaded with `MERGE`. Older mapping versions get `is_active = false`. Rows are never deleted.

## Not done here

No status mapping. No "Delayed" status, because there is no reliable target completion date.

## 1. Confirm upstream safety and select the mapping version

**What:** check that `silver_project` is one validated run and that CONFIG validation passed. Then select the single active approved DPWH status mapping version, if any.

**Why:** the mapping is read straight from configuration with `source_system = 'DPWH'`. Gold does not depend on the Silver component join for status. Gold trims the approved status keys to match Silver's trimmed `status_raw` (D-37).

**Protects:** more than one active approved version stops the run.

In [ ]:
%sql
USE CATALOG `buildabida-capstone`;
CREATE SCHEMA IF NOT EXISTS `03-gold`;

DECLARE OR REPLACE VARIABLE project_source_system STRING DEFAULT 'DPWH';
DECLARE OR REPLACE VARIABLE status_rule_version STRING DEFAULT 'gold-dim-project-status-v1';
DECLARE OR REPLACE VARIABLE current_status_mapping_version STRING;

SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT(DISTINCT run_id) = 1
    AND COUNT(DISTINCT source_snapshot_id) = 1
    AND COUNT(DISTINCT source_ingest_run_id) = 1
    AND COUNT(*) = COUNT(DISTINCT project_key),
    'STOP: silver_project must be nonempty and unique in one selected run'
)
FROM `02-silver`.silver_project;

SELECT ASSERT_TRUE(
    COUNT(DISTINCT dq.table_name) = 2
    AND COUNT_IF(dq.action = 'stop' AND dq.status IN ('FAIL', 'ERROR')) = 0,
    'STOP: selected Silver project validation is missing or unsafe'
)
FROM `04-validation`.silver_dq_results AS dq
WHERE dq.table_name IN ('silver_project', 'silver_dpwh_project_component')
  AND dq.pipeline_run_id = (SELECT MAX(project.run_id) FROM `02-silver`.silver_project AS project);

CREATE OR REPLACE TEMPORARY VIEW latest_config_validation AS
SELECT * EXCEPT (validation_rank)
FROM (
    SELECT *, DENSE_RANK() OVER (ORDER BY run_ts DESC, run_id DESC) AS validation_rank
    FROM `04-validation`.silver_config_dq_results
    WHERE layer = 'silver_config'
)
WHERE validation_rank = 1;

SELECT ASSERT_TRUE(
    COUNT(*) > 0 AND COUNT_IF(action = 'stop' AND status IN ('FAIL', 'ERROR')) = 0,
    'STOP: latest Silver configuration validation is missing or unsafe'
)
FROM latest_config_validation;

SELECT ASSERT_TRUE(
    COUNT(DISTINCT status_mapping_version) <= 1,
    'STOP: multiple active approved DPWH status mapping versions are unsafe'
)
FROM `02-silver`.config_project_status_mapping
WHERE source_system = project_source_system
  AND approval_status = 'APPROVED'
  AND is_active = TRUE;

CREATE OR REPLACE TEMPORARY VIEW project_load_lineage AS
SELECT run_id AS source_ingest_run_id, source_version,
       COALESCE(completed_at, started_at) AS source_load_ts
FROM (
    SELECT *, ROW_NUMBER() OVER (
        PARTITION BY run_id
        ORDER BY COALESCE(completed_at, started_at) DESC, started_at DESC
    ) AS audit_rank
    FROM `01-bronze`.load_log
    WHERE table_name = 'dpwh_projects'
      AND status IN ('SUCCESS', 'SKIPPED_IDEMPOTENT')
)
WHERE audit_rank = 1;

CREATE OR REPLACE TEMPORARY VIEW selected_status_versions AS
WITH project_context AS (
    SELECT
        MAX(run_id) AS project_run_id,
        MAX(source_ingest_run_id) AS project_source_ingest_run_id
    FROM `02-silver`.silver_project
),
mapping_context AS (
    SELECT COALESCE(MAX(status_mapping_version), 'NO_APPROVED_STATUS_MAPPING') AS status_mapping_version
    FROM `02-silver`.config_project_status_mapping
    WHERE source_system = project_source_system
      AND approval_status = 'APPROVED'
      AND is_active = TRUE
)
SELECT
    project_context.project_run_id,
    mapping_context.status_mapping_version,
    load.source_load_ts AS project_source_load_ts,
    SHA2(CONCAT_WS(
        '|', project_context.project_run_id, mapping_context.status_mapping_version, status_rule_version
    ), 256) AS status_run_id
FROM project_context
CROSS JOIN mapping_context
LEFT JOIN project_load_lineage AS load
    ON project_context.project_source_ingest_run_id = load.source_ingest_run_id;

SET VAR current_status_mapping_version = (
    SELECT status_mapping_version FROM selected_status_versions
);

CREATE OR REPLACE TEMPORARY VIEW approved_status_mappings AS
SELECT
    TRIM(mapping.source_status) AS source_status,
    mapping.standardized_status,
    mapping.status_group,
    mapping.status_mapping_version
FROM `02-silver`.config_project_status_mapping AS mapping
WHERE mapping.source_system = project_source_system
  AND mapping.approval_status = 'APPROVED'
  AND mapping.is_active = TRUE
  AND mapping.status_mapping_version = current_status_mapping_version;

SELECT ASSERT_TRUE(
    COUNT(*) = COUNT(DISTINCT source_status),
    'STOP: an approved status mapping version maps one trimmed source status more than once'
)
FROM approved_status_mappings;

## 2. Create the table

**What:** create `03-gold.dim_project_status` with the column names, types, and order in `docs/gold_model.md`.

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS `03-gold`.dim_project_status (
    status_key INT NOT NULL COMMENT 'Surrogate key. HASH of the label PROJECT_STATUS, the source system, source status, and mapping version. Key 0 is a missing or conflicting status.',
    source_status STRING COMMENT 'DPWH status exactly as published, such as Completed or On-Going.',
    standardized_status STRING COMMENT 'Team-approved standardized status. NULL until a status mapping is approved.',
    status_group STRING COMMENT 'Team-approved status group. NULL until a status mapping is approved.',
    status_mapping_version STRING COMMENT 'Approved status mapping version. NO_APPROVED_STATUS_MAPPING when none exists.',
    source_system STRING COMMENT 'Source system of the status. Always DPWH.',
    is_active BOOLEAN COMMENT 'True for rows of the current mapping version. Older versions stay for history.',
    run_id STRING COMMENT 'Deterministic Gold run ID. Hash of the Silver project run, the mapping version, and the Gold rule version.',
    source_load_ts TIMESTAMP COMMENT 'Bronze load time of the DPWH snapshot behind these statuses.'
)
USING DELTA
COMMENT 'Gold project status dimension. One DPWH source status per mapping version plus key 0 for a missing status. Standardized values appear only after team approval.';

## 3. Build and merge the statuses

**What:** combine the observed and approved statuses, attach the approved mapping, add key `0`, and merge by key. Then mark older mapping versions inactive.

**Why:** the key is `HASH` of the source system, status, and mapping version, so it is the same on every run. When the team approves a new version, every status gets a new row and the old rows stay as history.

**Expected validation:** keys are unique, and exactly one row has key `0`.

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW dim_project_status_source AS
WITH status_values AS (
    SELECT project.status_raw AS source_status
    FROM `02-silver`.silver_project AS project
    WHERE project.status_raw IS NOT NULL
    -- union on purpose: an approved status that is also observed is one member
    UNION
    SELECT mapping.source_status
    FROM approved_status_mappings AS mapping
),
known_statuses AS (
    SELECT
        HASH(CONCAT_WS(
            '|', 'PROJECT_STATUS', project_source_system, status_values.source_status, selected.status_mapping_version
        )) AS status_key,
        status_values.source_status,
        mapping.standardized_status,
        mapping.status_group,
        selected.status_mapping_version
    FROM status_values
    CROSS JOIN selected_status_versions AS selected
    LEFT JOIN approved_status_mappings AS mapping
        ON status_values.source_status = mapping.source_status
),
unknown_status AS (
    SELECT
        0 AS status_key,
        'Missing or conflicting status' AS source_status,
        CAST(NULL AS STRING) AS standardized_status,
        CAST(NULL AS STRING) AS status_group,
        selected.status_mapping_version
    FROM selected_status_versions AS selected
),
all_statuses AS (
    SELECT * FROM unknown_status
    UNION ALL
    SELECT * FROM known_statuses
)
SELECT
    all_statuses.*,
    project_source_system AS source_system,
    TRUE AS is_active,
    selected.status_run_id AS run_id,
    selected.project_source_load_ts AS source_load_ts
FROM all_statuses
CROSS JOIN selected_status_versions AS selected;

SELECT ASSERT_TRUE(
    COUNT(*) = COUNT(DISTINCT status_key)
    AND COUNT_IF(status_key = 0) = 1,
    'STOP: dim_project_status keys are not unique or a status collides with key 0'
)
FROM dim_project_status_source;

MERGE INTO `03-gold`.dim_project_status AS target
USING dim_project_status_source AS source
ON target.status_key = source.status_key
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

UPDATE `03-gold`.dim_project_status
SET is_active = FALSE
WHERE status_key <> 0
  AND status_mapping_version <> current_status_mapping_version;

# Summary

Created or refreshed `03-gold.dim_project_status`. Every observed DPWH status has a key. Standardized values stay `NULL` until the team approves a status mapping. Key `0` covers missing or conflicting statuses.

Coverage: the query below lists every active status. Known limits: no status mapping is approved, so `standardized_status` and `status_group` are `NULL`. Delivery measures stay blocked until a mapping is approved. About 30 statuses are shifted CSV values, and the Gold validator will flag them.

Next: `04_gold_dim_project`.

In [ ]:
%sql
SELECT
    status_key,
    source_status,
    standardized_status,
    status_group,
    status_mapping_version,
    is_active
FROM `03-gold`.dim_project_status
WHERE is_active
ORDER BY status_key = 0 DESC, source_status;